## 07 — Cross-source match & team ID mapping

Sportmonks and FotMob use separate ID systems — the same match has different IDs in each source. This notebook creates the bridge table `silver.match_source_map` linking `sportmonks_fixture_id` to `fotmob_match_id`. The mapping is manual (Excel file) because no shared global match identifier exists.


### 1. Load manual mapping from Excel / Wczytanie ręcznego mapowania z Excela

**PL:** Instalujemy `openpyxl` (wymagane przez pandas do odczytu xlsx). Wczytujemy plik mapowania z Unity Catalog Volume. Plik zawiera ręcznie dopasowane pary ID z obu źródeł plus metadane meczu.  
**EN:** Installs `openpyxl` (required by pandas to read xlsx). Reads the mapping file from the Unity Catalog Volume. The file contains manually matched ID pairs from both sources plus match metadata.


In [ ]:
pip install openpyxl

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
import pandas as pd
from pyspark.sql import functions as F

MATCHING_PATH = (
    "/Volumes/wsl_analytics/reference/"
    "manual_mappings/matching.xlsx"
)

matching_pd = pd.read_excel(MATCHING_PATH, engine='openpyxl')

display(matching_pd)

fotmob_match_id,league_id,match_datetime,match_date,home_team_id,away_team_id,home_team,away_team,fixture_id
4892931,9227,2025-09-05T18:30:00.000Z,2025-09-05T00:00:00.000Z,258661,231488,Chelsea,Manchester City,19499028
4892932,9227,2025-09-06T12:30:00.000Z,2025-09-06T00:00:00.000Z,258657,1075419,Arsenal,London City Lionesses,19499029
4892933,9227,2025-09-07T11:00:00.000Z,2025-09-07T00:00:00.000Z,258665,258663,Liverpool,Everton,19499033
4892934,9227,2025-09-07T11:00:00.000Z,2025-09-07T00:00:00.000Z,231505,231494,Brighton,Aston Villa,19499032
4892935,9227,2025-09-07T11:00:00.000Z,2025-09-07T00:00:00.000Z,954396,614954,Manchester United,Leicester City,19499030
4892936,9227,2025-09-07T11:00:00.000Z,2025-09-07T00:00:00.000Z,628117,231497,Tottenham Hotspur,West Ham United,19499031
4892969,9227,2025-09-12T18:30:00.000Z,2025-09-12T00:00:00.000Z,231488,231505,Manchester City,Brighton,19499034
4892971,9227,2025-09-12T18:30:00.000Z,2025-09-12T00:00:00.000Z,231497,258657,West Ham United,Arsenal,19499035
4892970,9227,2025-09-14T11:00:00.000Z,2025-09-14T00:00:00.000Z,1075419,954396,London City Lionesses,Manchester United,19499036
4892972,9227,2025-09-14T11:00:00.000Z,2025-09-14T00:00:00.000Z,231494,258661,Aston Villa,Chelsea,19499038


### 2. Convert to Spark 

Converts the pandas DataFrame to a Spark DataFrame for joining with other Unity Catalog tables.


In [ ]:
matching_df = spark.createDataFrame(matching_pd)

display(matching_df)

fotmob_match_id,league_id,match_datetime,match_date,home_team_id,away_team_id,home_team,away_team,fixture_id
4892931,9227,2025-09-05T18:30:00.000Z,2025-09-05T00:00:00.000Z,258661,231488,Chelsea,Manchester City,19499028
4892932,9227,2025-09-06T12:30:00.000Z,2025-09-06T00:00:00.000Z,258657,1075419,Arsenal,London City Lionesses,19499029
4892933,9227,2025-09-07T11:00:00.000Z,2025-09-07T00:00:00.000Z,258665,258663,Liverpool,Everton,19499033
4892934,9227,2025-09-07T11:00:00.000Z,2025-09-07T00:00:00.000Z,231505,231494,Brighton,Aston Villa,19499032
4892935,9227,2025-09-07T11:00:00.000Z,2025-09-07T00:00:00.000Z,954396,614954,Manchester United,Leicester City,19499030
4892936,9227,2025-09-07T11:00:00.000Z,2025-09-07T00:00:00.000Z,628117,231497,Tottenham Hotspur,West Ham United,19499031
4892969,9227,2025-09-12T18:30:00.000Z,2025-09-12T00:00:00.000Z,231488,231505,Manchester City,Brighton,19499034
4892971,9227,2025-09-12T18:30:00.000Z,2025-09-12T00:00:00.000Z,231497,258657,West Ham United,Arsenal,19499035
4892970,9227,2025-09-14T11:00:00.000Z,2025-09-14T00:00:00.000Z,1075419,954396,London City Lionesses,Manchester United,19499036
4892972,9227,2025-09-14T11:00:00.000Z,2025-09-14T00:00:00.000Z,231494,258661,Aston Villa,Chelsea,19499038


### 3. Schema check 

Checks data types — in particular that IDs are numeric and dates are in the correct format.


In [ ]:
matching_df.printSchema()

root
 |-- fotmob_match_id: long (nullable = true)
 |-- league_id: long (nullable = true)
 |-- match_datetime: string (nullable = true)
 |-- match_date: timestamp (nullable = true)
 |-- home_team_id: long (nullable = true)
 |-- away_team_id: long (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- fixture_id: long (nullable = true)



### 4. Build match_source_map base 

Casts column types (`long` for IDs, `timestamp`/`date` for dates) and adds `mapping_method = "manual"` to allow future distinction between manual and potential automated mappings.


In [ ]:
match_source_map_df = (
    matching_df
    .select(

        F.col("fixture_id")
            .cast("long")
            .alias("sportmonks_fixture_id"),

        F.col("fotmob_match_id")
            .cast("long")
            .alias("fotmob_match_id"),

        F.col("league_id")
            .cast("long")
            .alias("fotmob_league_id"),

        F.to_timestamp(
            "match_datetime"
        ).alias("match_datetime"),

        F.to_date(
            "match_date"
        ).alias("match_date"),

        F.col("home_team_id")
            .cast("long")
            .alias("fotmob_home_team_id"),

        F.col("away_team_id")
            .cast("long")
            .alias("fotmob_away_team_id"),

        F.col("home_team")
            .alias("fotmob_home_team"),

        F.col("away_team")
            .alias("fotmob_away_team")
    )

    .withColumn(
        "mapping_method",
        F.lit("manual")
    )
)

### 5. Preview 

Displays the mapping table.


In [ ]:
display(match_source_map_df)

sportmonks_fixture_id,fotmob_match_id,fotmob_league_id,match_datetime,match_date,fotmob_home_team_id,fotmob_away_team_id,fotmob_home_team,fotmob_away_team,mapping_method
19499028,4892931,9227,2025-09-05T18:30:00.000Z,2025-09-05,258661,231488,Chelsea,Manchester City,manual
19499029,4892932,9227,2025-09-06T12:30:00.000Z,2025-09-06,258657,1075419,Arsenal,London City Lionesses,manual
19499033,4892933,9227,2025-09-07T11:00:00.000Z,2025-09-07,258665,258663,Liverpool,Everton,manual
19499032,4892934,9227,2025-09-07T11:00:00.000Z,2025-09-07,231505,231494,Brighton,Aston Villa,manual
19499030,4892935,9227,2025-09-07T11:00:00.000Z,2025-09-07,954396,614954,Manchester United,Leicester City,manual
19499031,4892936,9227,2025-09-07T11:00:00.000Z,2025-09-07,628117,231497,Tottenham Hotspur,West Ham United,manual
19499034,4892969,9227,2025-09-12T18:30:00.000Z,2025-09-12,231488,231505,Manchester City,Brighton,manual
19499035,4892971,9227,2025-09-12T18:30:00.000Z,2025-09-12,231497,258657,West Ham United,Arsenal,manual
19499036,4892970,9227,2025-09-14T11:00:00.000Z,2025-09-14,1075419,954396,London City Lionesses,Manchester United,manual
19499038,4892972,9227,2025-09-14T11:00:00.000Z,2025-09-14,231494,258661,Aston Villa,Chelsea,manual


### 6. Load silver.fixture_teams 

Reads the Sportmonks team table to add Sportmonks team IDs to the mapping (they're not in the Excel file — they come from silver).


In [ ]:
fixture_teams_df = spark.table(
    "wsl_analytics.silver.fixture_teams"
)

### 7. Prepare home teams 

Filters `fixture_teams` to home matches and selects `sportmonks_fixture_id`, `sportmonks_home_team_id`, and `sportmonks_home_team`. Will join on `sportmonks_fixture_id`.


In [ ]:
sportmonks_home_df = (
    fixture_teams_df
    .filter(
        F.col("location") == "home"
    )
    .select(

        F.col("fixture_id")
            .alias("sportmonks_fixture_id"),

        F.col("team_id")
            .alias("sportmonks_home_team_id"),

        F.col("team_name")
            .alias("sportmonks_home_team")
    )
)

### 8. Prepare away teams 

Same as above for away teams.


In [ ]:
sportmonks_away_df = (
    fixture_teams_df
    .filter(
        F.col("location") == "away"
    )
    .select(

        F.col("fixture_id")
            .alias("sportmonks_fixture_id"),

        F.col("team_id")
            .alias("sportmonks_away_team_id"),

        F.col("team_name")
            .alias("sportmonks_away_team")
    )
)

### 9. Enrich mapping with Sportmonks team IDs 

Two left joins add Sportmonks team IDs (home and away) to the mapping table. Left join preserves matches even if team data is missing.


In [ ]:
match_source_map_df = (
    match_source_map_df

    .join(
        sportmonks_home_df,
        on="sportmonks_fixture_id",
        how="left"
    )

    .join(
        sportmonks_away_df,
        on="sportmonks_fixture_id",
        how="left"
    )
)

### 10. Preview cross-source name alignment 

Displays team names from both sources side by side — manual verification that the mapping is correct (e.g. "Arsenal Women" in Sportmonks = "Arsenal" in FotMob).


In [ ]:
display(
    match_source_map_df.select(
        "sportmonks_fixture_id",
        "fotmob_match_id",

        "sportmonks_home_team",
        "fotmob_home_team",

        "sportmonks_away_team",
        "fotmob_away_team"
    )
)

sportmonks_fixture_id,fotmob_match_id,sportmonks_home_team,fotmob_home_team,sportmonks_away_team,fotmob_away_team
19499028,4892931,Chelsea W,Chelsea,Manchester City W,Manchester City
19499029,4892932,Arsenal W,Arsenal,London City Lionesses W,London City Lionesses
19499033,4892933,Liverpool W,Liverpool,Everton W,Everton
19499032,4892934,Brighton W,Brighton,Aston Villa W,Aston Villa
19499030,4892935,Manchester United W,Manchester United,Leicester W,Leicester City
19499031,4892936,Tottenham W,Tottenham Hotspur,West Ham W,West Ham United
19499034,4892969,Manchester City W,Manchester City,Brighton W,Brighton
19499035,4892971,West Ham W,West Ham United,Arsenal W,Arsenal
19499036,4892970,London City Lionesses W,London City Lionesses,Manchester United W,Manchester United
19499038,4892972,Aston Villa W,Aston Villa,Chelsea W,Chelsea


### 11. Select final columns 

Selects and orders columns for the final mapping table.


In [ ]:
match_source_map_df = (
    match_source_map_df
    .select(

        "sportmonks_fixture_id",
        "fotmob_match_id",

        "match_date",
        "match_datetime",

        "sportmonks_home_team_id",
        "fotmob_home_team_id",

        "sportmonks_home_team",
        "fotmob_home_team",

        "sportmonks_away_team_id",
        "fotmob_away_team_id",

        "sportmonks_away_team",
        "fotmob_away_team",

        "mapping_method"
    )
)

### 12. Preview final 

Final preview of the mapping table before writing.


In [ ]:
display(match_source_map_df)

sportmonks_fixture_id,fotmob_match_id,match_date,match_datetime,sportmonks_home_team_id,fotmob_home_team_id,sportmonks_home_team,fotmob_home_team,sportmonks_away_team_id,fotmob_away_team_id,sportmonks_away_team,fotmob_away_team,mapping_method
19499028,4892931,2025-09-05,2025-09-05T18:30:00.000Z,59583,258661,Chelsea W,Chelsea,59584,231488,Manchester City W,Manchester City,manual
19499029,4892932,2025-09-06,2025-09-06T12:30:00.000Z,62847,258657,Arsenal W,Arsenal,132782,1075419,London City Lionesses W,London City Lionesses,manual
19499033,4892933,2025-09-07,2025-09-07T11:00:00.000Z,132874,258665,Liverpool W,Liverpool,132784,258663,Everton W,Everton,manual
19499032,4892934,2025-09-07,2025-09-07T11:00:00.000Z,132716,231505,Brighton W,Brighton,132710,231494,Aston Villa W,Aston Villa,manual
19499030,4892935,2025-09-07,2025-09-07T11:00:00.000Z,228655,954396,Manchester United W,Manchester United,228634,614954,Leicester W,Leicester City,manual
19499031,4892936,2025-09-07,2025-09-07T11:00:00.000Z,132699,628117,Tottenham W,Tottenham Hotspur,228102,231497,West Ham W,West Ham United,manual
19499034,4892969,2025-09-12,2025-09-12T18:30:00.000Z,59584,231488,Manchester City W,Manchester City,132716,231505,Brighton W,Brighton,manual
19499035,4892971,2025-09-12,2025-09-12T18:30:00.000Z,228102,231497,West Ham W,West Ham United,62847,258657,Arsenal W,Arsenal,manual
19499036,4892970,2025-09-14,2025-09-14T11:00:00.000Z,132782,1075419,London City Lionesses W,London City Lionesses,228655,954396,Manchester United W,Manchester United,manual
19499038,4892972,2025-09-14,2025-09-14T11:00:00.000Z,132710,231494,Aston Villa W,Aston Villa,59583,258661,Chelsea W,Chelsea,manual


### 13. Write silver.match_source_map 

Writes the match mapping table.


In [ ]:
(
    match_source_map_df
    .write
    .mode("overwrite")
    .saveAsTable(
        "wsl_analytics.silver.match_source_map"
    )
)

### 14. SQL verify 

Orders by match date — mapping completeness verification.


In [ ]:
%sql

SELECT *
FROM wsl_analytics.silver.match_source_map
ORDER BY match_date;

sportmonks_fixture_id,fotmob_match_id,match_date,match_datetime,sportmonks_home_team_id,fotmob_home_team_id,sportmonks_home_team,fotmob_home_team,sportmonks_away_team_id,fotmob_away_team_id,sportmonks_away_team,fotmob_away_team,mapping_method
19499028,4892931,2025-09-05,2025-09-05T18:30:00.000Z,59583,258661,Chelsea W,Chelsea,59584,231488,Manchester City W,Manchester City,manual
19499029,4892932,2025-09-06,2025-09-06T12:30:00.000Z,62847,258657,Arsenal W,Arsenal,132782,1075419,London City Lionesses W,London City Lionesses,manual
19499033,4892933,2025-09-07,2025-09-07T11:00:00.000Z,132874,258665,Liverpool W,Liverpool,132784,258663,Everton W,Everton,manual
19499030,4892935,2025-09-07,2025-09-07T11:00:00.000Z,228655,954396,Manchester United W,Manchester United,228634,614954,Leicester W,Leicester City,manual
19499031,4892936,2025-09-07,2025-09-07T11:00:00.000Z,132699,628117,Tottenham W,Tottenham Hotspur,228102,231497,West Ham W,West Ham United,manual
19499032,4892934,2025-09-07,2025-09-07T11:00:00.000Z,132716,231505,Brighton W,Brighton,132710,231494,Aston Villa W,Aston Villa,manual
19499035,4892971,2025-09-12,2025-09-12T18:30:00.000Z,228102,231497,West Ham W,West Ham United,62847,258657,Arsenal W,Arsenal,manual
19499034,4892969,2025-09-12,2025-09-12T18:30:00.000Z,59584,231488,Manchester City W,Manchester City,132716,231505,Brighton W,Brighton,manual
19499037,4892973,2025-09-14,2025-09-14T13:30:00.000Z,132784,258663,Everton W,Everton,132699,628117,Tottenham W,Tottenham Hotspur,manual
19499036,4892970,2025-09-14,2025-09-14T11:00:00.000Z,132782,1075419,London City Lionesses W,London City Lionesses,228655,954396,Manchester United W,Manchester United,manual


### 15-17. Build silver.team_source_map 

From `match_source_map`, builds `team_source_map` — a dictionary linking `sportmonks_team_id` to `fotmob_team_id`. Extracts (Sportmonks, FotMob) pairs separately for home and away teams, combines via `unionByName`, and deduplicates.


In [ ]:
home_team_map_df = (
    match_source_map_df
    .select(

        F.col("sportmonks_home_team_id")
            .alias("sportmonks_team_id"),

        F.col("fotmob_home_team_id")
            .alias("fotmob_team_id"),

        F.col("sportmonks_home_team")
            .alias("sportmonks_team_name"),

        F.col("fotmob_home_team")
            .alias("fotmob_team_name")
    )
)

In [ ]:
away_team_map_df = (
    match_source_map_df
    .select(

        F.col("sportmonks_away_team_id")
            .alias("sportmonks_team_id"),

        F.col("fotmob_away_team_id")
            .alias("fotmob_team_id"),

        F.col("sportmonks_away_team")
            .alias("sportmonks_team_name"),

        F.col("fotmob_away_team")
            .alias("fotmob_team_name")
    )
)

In [ ]:
team_source_map_df = (
    home_team_map_df
    .unionByName(
        away_team_map_df
    )
    .dropDuplicates([
        "sportmonks_team_id",
        "fotmob_team_id"
    ])
)

In [ ]:
display(team_source_map_df)

sportmonks_team_id,fotmob_team_id,sportmonks_team_name,fotmob_team_name
132699,628117,Tottenham W,Tottenham Hotspur
59583,258661,Chelsea W,Chelsea
132874,258665,Liverpool W,Liverpool
132784,258663,Everton W,Everton
132716,231505,Brighton W,Brighton
228102,231497,West Ham W,West Ham United
62847,258657,Arsenal W,Arsenal
228634,614954,Leicester W,Leicester City
59584,231488,Manchester City W,Manchester City
132782,1075419,London City Lionesses W,London City Lionesses


### 18. Write silver.team_source_map 

Writes the team mapping table.


In [ ]:
(
    team_source_map_df
    .write
    .mode("overwrite")
    .saveAsTable(
        "wsl_analytics.silver.team_source_map"
    )
)